# MA12003 Programming coursework 

**Before making any changes to this notebook, save it as `coursework_USERNAME.ipynb` where `USERNAME` is your username. You need to save this new file in the `ma12003_workspace` directory.**

**DO NOT create any extra cells in this notebook** with the exception of additional cells under the Additional Examples section at the bottom of this file.

* Your code should be put in the relevant cell. 

* Any packages that you need to import should be put in the section titled **Imports**. 

* Code for your implementation of the required classes should appear in the appropriate cells in the section titled **Classes**. Running these cells **should not** produce any printed output.

* The examples provided in Coursework should be put in the relevant cells, e.g. Task 1.a Example cell, Task 1.b Example cell, etc. In particular the examples will produce printed output and **should not** be put in the cells for the classes. 

* Any additional tests that you ran for checking correctness of your code can be put under the **Additional Examples** section. Note that this is the only section under which you are allowed to add new cells. Note that all cells under **Additional Examples** section will be **ignored for the purpose of assessment**. The only reason to include these is to provide **hints** to examiner in case your code seems to be failing basic tests due to simple errors. However there is no guarantee that these will be reviewed. 

#### Enter your full name in the following cell:

Ewan Brett

## **Imports**

* All your imported libraries should be imported in the following cell. 

* **If you import more libraries later in your code, you will be penalised.**

In [1]:
from abc import ABC, abstractmethod
import random
import string
#import good marks

## **Classes**

### **`PseudometricSpace`** class

Copy the skeletal code for the abstract class `PseudometricSpace` in the cell below, and modify it as required.

In [2]:
class PseudometricSpace(ABC):
    """
    Abstract class for pseudometric spaces.
    """
    
    # don't change this
    @abstractmethod
    def dist(self, other):
        ...

    # don't change this
    @abstractmethod
    def random():
        ...

    # don't change this
    def __repr__(self):
        return str(self)

    # Task 2
    def test_axioms(self):
        epsilon = 1e-10
        test_dictionary = {
            "nonnegativity": True,
            "reflexivity": True,
            "symmetry": True,
            "triangle_inequality": True
        }
        for i in range(1,101):
            x = type(self).random()
            y = type(self).random()
            z = type(self).random()
            if x.dist(y) < -epsilon:
                test_dictionary["nonnegativity"] = False
            elif (x.dist(x) < -epsilon) or (x.dist(x) > epsilon):
                test_dictionary["reflexivity"] = False
            elif abs(x.dist(y) - y.dist(x)) > epsilon:
                test_dictionary["symmetry"] = False
            elif x.dist(y) + y.dist(z) - x.dist(z) < -epsilon:
                test_dictionary["triangle_inequality"] = False
        return test_dictionary
                
    # Task 3.a
    def total_distance(self, points):
        s = set(points)
        totaldist = 0
        for x in s:
            totaldist += self.dist(x)
        return totaldist

    # Task 3.b
    def median(points):
        set_points = set(points)
        list_points = list(points)
        d = len(list_points)
        if d == 0:
            raise Exception("Empty set has no median")
        distance = list_points[0].total_distance(set_points)
        median = list_points[0]
        for i in range(1,d):
            x = list_points[i].total_distance(set_points)
            if x <= distance:
                distance = x
                median = list_points[i]
        return median

    # Task 3.c
    def partition(x1, x2, points):
        list_points = list(points)
        d = len(list_points)
        Y1 = [f for f in list_points if (f == x1) or (f != x2 and f.dist(x1) < f.dist(x2))]
        Y2 = [f for f in list_points if (f == x2) or (f != x1 and f.dist(x2) <= f.dist(x1))]
        set_Y1 = set(Y1)
        set_Y2 = set(Y2)
        partitions = [set_Y1,set_Y2]
        return partitions

    # Task 4
    def clusters(x1, x2, points):
        x = type(x1)
        clusters = x.partition(x1,x2,points)
        Y1 = clusters[0]
        Y2 = clusters[1] 
        Y1_med = x.median(Y1)
        Y2_med = x.median(Y2)
        centroid_guesses = [{x1,x2}]
        centroids = [Y1_med,Y2_med]
        while not {Y1_med,Y2_med} in centroid_guesses:
            clusters = x.partition(Y1_med, Y2_med, points)
            centroid_guesses.append({Y1_med,Y2_med})
            Y1_med = x.median(clusters[0])
            Y2_med = x.median(clusters[1])
            centroids = [Y1_med,Y2_med]
        return centroids , clusters , centroid_guesses

### **`RealNumber`** class

Copy the skeletal code for the concrete class `RealNumber` in the cell below, and modify it as required. 

In [3]:
class RealNumber(PseudometricSpace):

    # don't change this
    def __init__(self, value):
        """
        Initialize an instance of RealNumber 
        where value is a number. For example:

        x = RealNumber(3.14)
        """
        self.value = value

    def __str__(self):
        return str(self.value)

    def dist(self, other):
        return abs(self.value - other.value)

    def random():
        random_RealNumber = random.randint(0,100) /4
        return RealNumber(random_RealNumber)

### **`String`** class

Copy the skeletal code for the concrete class `String` in the cell below, and modify it as required. 

In [4]:
class String(PseudometricSpace):

    # don't change this
    def __init__(self, text):
        """
        Initialize an instance of String
        where text is a string. For example:

        s = String("hello")
        """
        self.text = text

    def __str__(self):
        return str(self.text)

    def dist(self,other):
        return abs(len(self.text)-len(other.text))

    def random():
        length = random.randint(1,10)
        random_String = ''.join(random.choices(string.ascii_lowercase, k=length))
        return String(random_String)

### **`ManhattanPoint`** class

Copy the skeletal code for the concrete class `ManhattanPoint` in the cell below, and modify it as required. 

In [5]:
class ManhattanPoint(PseudometricSpace):

    # don't change this
    def __init__(self, s, x):
        """
        Initialize an instance of ManhattanPoint,
        where 's' is an instance of String
        and 'x' is an instance of RealNumber.
        
        For example:

        s = String("hello")
        x = RealNumber(3.14)
        m = ManhattanPoint(s, r)
        """
        self.s = s
        self.x = x

    def __str__(self):
        return "(" + str(self.s) + ", " + str(self.x) + ")"

    def dist(self,other):
        return String.dist(self.s,other.s) + RealNumber.dist(self.x,other.x)

    def random():
        return ManhattanPoint(String.random(),RealNumber.random())

## **Example Runs**

## Task 1

### Task 1.a Example

Copy the code for Task 1.a Example in the cell below and test to see if it has the required behaviour

In [6]:
x1 = RealNumber(7.5)
s1 = String("hello")
m1 = ManhattanPoint(s1, x1)

print(str(x1))
print(str(s1))
print(str(m1))

7.5
hello
(hello, 7.5)


### Task 1.b Example

Copy the code for Task 1.b Example in the cell below and test to see if it has the required behaviour

In [7]:
x2 = RealNumber(2.7)
s2 = String("outer space")
m2 = ManhattanPoint(s2, x2)

print(x1.dist(x2))
print(s1.dist(s2))
print(m1.dist(m2))

4.8
6
10.8


### Task 1.c Example

Copy the code for Task 1.c Example in the cell below and test to see if it has the required behaviour

In [8]:
print(RealNumber.random())
print(String.random())
print(ManhattanPoint.random())

9.75
ccha
(lkqh, 23.0)


## Task 2

### Task 2 Example

Copy the code for Task 2 Example in the cell below and test to see if it has the required behaviour

In [9]:
r = RealNumber(0)
print(r.test_axioms())

class TrivialSpace(PseudometricSpace):
    def dist(self, other):
        return 1

    # Task 1.c
    def random():
        return TrivialSpace()

t = TrivialSpace()
print(t.test_axioms())

{'nonnegativity': True, 'reflexivity': True, 'symmetry': True, 'triangle_inequality': True}
{'nonnegativity': True, 'reflexivity': False, 'symmetry': True, 'triangle_inequality': True}


## Task 3

### Task 3.a Example

Copy the code for Task 3.a Example in the cell below and test to see if it has the required behaviour

In [10]:
clist = [-5/4, 2/3, 5/6, 1., 1/2, -3/4, 3/2, 8/7, 7/4, 3., -1/2, -1., -3/2, 4/3, 5/4]
wlist = ["cat", "dog", "fish", "bird", "elephant", "giraffe", "lion", "tiger", "bear", "wolf", "fox", "whale", "shark", "eagle", "hawk"]

rlist = [RealNumber(c) for c in clist]
slist = [String(c) for c in wlist]
mlist = [ManhattanPoint(s, r) for (s, r) in list(zip(slist, rlist))]

print(rlist[5].total_distance(set(rlist)))
print(slist[5].total_distance(set(slist)))
print(mlist[5].total_distance(set(mlist)))

22.226190476190478
39
61.22619047619048


### Task 3.b Example

Copy the code for Task 3.b Example in the cell below and test to see if it has the required behaviour

In [11]:
print(RealNumber.median(set(rlist)))
print(String.median(set(slist)))
print(ManhattanPoint.median(set(mlist)))

0.8333333333333334
hawk
(fish, 0.8333333333333334)


### Task 3.c Example

Copy the code for Task 3.c Example in the cell below and test to see if it has the required behaviour

In [12]:
for points in [rlist, slist, mlist]:
    x1, x2 = points[0], points[1]
    ClassName = type(x1)
    p1, p2 = ClassName.partition(x1, x2, set(points))
    print(f"Closest to {x1}: {p1}")
    print(f"Closest to {x2}: {p2}")
    print("")

Closest to -1.25: {-1.25, -1.5, -0.75, -1.0, -0.5}
Closest to 0.6666666666666666: {0.8333333333333334, 1.75, 1.25, 1.1428571428571428, 1.0, 1.3333333333333333, 0.5, 0.6666666666666666, 1.5, 3.0}

Closest to cat: {cat}
Closest to dog: {bear, shark, wolf, bird, dog, lion, tiger, elephant, fox, fish, hawk, giraffe, eagle, whale}

Closest to (cat, -1.25): {(fox, -0.5), (cat, -1.25), (giraffe, -0.75), (whale, -1.0), (shark, -1.5)}
Closest to (dog, 0.6666666666666666): {(bird, 1.0), (dog, 0.6666666666666666), (hawk, 1.25), (wolf, 3.0), (fish, 0.8333333333333334), (eagle, 1.3333333333333333), (tiger, 1.1428571428571428), (bear, 1.75), (lion, 1.5), (elephant, 0.5)}



## Task 4

### Task 4 Example

Copy the code for Task 4 Example in the cell below and test to see if it has the required behaviour

In [13]:
for points in [rlist, slist, mlist]:
    ClassName = type(points[0])
    centroids, clusters, centroid_guesses = ClassName.clusters(points[0], points[1], set(points))

    print(f"centroid 1: {centroids[0]}")
    print(f"cluster 1: {clusters[0]}")
    print(f"centroid 2: {centroids[1]}")
    print(f"cluster 2: {clusters[1]}")
    print("")

centroid 1: -1.0
cluster 1: {-1.25, -1.5, -0.75, -1.0, -0.5}
centroid 2: 1.1428571428571428
cluster 2: {0.8333333333333334, 1.75, 1.25, 1.1428571428571428, 1.0, 1.3333333333333333, 0.5, 0.6666666666666666, 1.5, 3.0}

centroid 1: dog
cluster 1: {fox, cat, dog}
centroid 2: whale
cluster 2: {bear, shark, wolf, bird, lion, tiger, elephant, fish, hawk, giraffe, eagle, whale}

centroid 1: (whale, -1.0)
cluster 1: {(fox, -0.5), (cat, -1.25), (giraffe, -0.75), (whale, -1.0), (shark, -1.5), (elephant, 0.5)}
centroid 2: (hawk, 1.25)
cluster 2: {(bird, 1.0), (dog, 0.6666666666666666), (hawk, 1.25), (wolf, 3.0), (fish, 0.8333333333333334), (eagle, 1.3333333333333333), (tiger, 1.1428571428571428), (bear, 1.75), (lion, 1.5)}



## Additional Examples

* Any additional examples that you test your code with **can** be provided below. 

* Note that these are **NOT** a required part of the submission and **WILL NOT BE ASSESSED**. For this reason, if you prefer, you could also delete the examples below before submission and only use the space below for your own tests.

* Feel free to add more cells below as required. Reminder: The cells below will be ignored during assessment, with the possible exception of identifying simple errors.